---

### 🎓 **Professor**: Apostolos Filippas

### 📘 **Class**: AI Engineering

### 📋 **Topic**: You Can Just Build Things

🚫 **Note**: You are not allowed to share the contents of this notebook with anyone outside this class without written permission by the professor.

---

## Welcome!

In our firstfour lectures, we've covered how
1. We can call LLMs via APIs and get structured responses
2. We can build lexical search with BM25
3. We can build semantic search with embeddings
4. We can combine lexical and semantic search into hybrid search

Today you will put it all together by building a Retrieval Augmented Generation (RAG) system.
- This is a question-answering bot that can answer questions about Fordham University
- You will use real data scraped from the Fordham website.


Your RAG pipeline will look like this:

```
User Question
     ↓
1. RETRIEVE: Find relevant documents (search!)
     ↓
2. AUGMENT: Stuff those documents into a prompt
     ↓
3. GENERATE: Ask an LLM to answer using the context
     ↓
Answer
```


---

# 1. Look at your data

In `data/fordham-website.zip` you'll find **~9,500 Markdown files** scraped from Fordham's website. Each file is one page — admissions info, program descriptions, faculty pages, financial aid, campus life, and more.

Your task: **look at the data**
- The first step in any AI engineering or data science project should always be to familiarize yourself with the data.
- I cannot stress this enough.. without this step, it's hard to build anything useful.

Tips:
- Unzip the archive and look at some of the files. 
- Open a few in a text editor. 
- Get a feel for what you're working with.
- The first line of every file is always the **URL** of the page it was scraped from. The rest is the page content converted to Markdown. Here's an example — `gabelli-school-of-business_veterans.md`:

```markdown
https://www.fordham.edu/gabelli-school-of-business/veterans

# Military Veterans & Active Duty Members of the Military

## Transform Your Knowledge & Skills Into a Business Career for the Future

As a veteran or an active duty member of the United States Armed Services,
you have gained or are currently acquiring the invaluable organizational,
leadership, analytics, and technical knowledge and skills that hiring
managers seek. These transferrable skills provide a major advantage in
emerging, business-related industries where innovation, a global mind-set,
and the ability to lead individuals and teams in the continuously evolving
work environment, are critical for success.

By completing a graduate or undergraduate business degree at the Gabelli
School of Business, you can prepare for a lifelong career in some of
today's fastest-growing fields. ...

### Study at a Top-Ranked, Military-Friendly University

The Gabelli School of Business is part of Fordham University, the only
New York City university to be among those ranked "Best for Vets" by
Military Times. ...

### Learn How the Yellow Ribbon Program Works

The Yellow Ribbon GI Education Enhancement Program, or the Yellow Ribbon
Program, is a part of the Post-9/11 Veterans Educational Assistance Act
of 2008. ...
```

The filenames mirror the URL structure — underscores replace path separators (e.g. `gabelli-school-of-business_veterans.md` came from `/gabelli-school-of-business/veterans`). Some files are short (a few lines), others are quite long.

- Once you've looked around, load the files into Python. Python's built-in `zipfile` module can read zip archives without extracting to disk. Load them into a list of dictionaries or a DataFrame with at least two fields: the filename (or a clean page name) and the content

In [ ]:
import zipfile
import pandas as pd
import os

# Path to the zip file
zip_path = os.path.join("..", "data", "fordham-website.zip")

# Read all .md files from the zip into a list of dicts
documents = []

with zipfile.ZipFile(zip_path, "r") as zf:
    for name in zf.namelist():
        if name.endswith(".md"):
            content = zf.read(name).decode("utf-8")
            # The first line of every file is the URL
            lines = content.split("\n", 1)
            url = lines[0].strip()
            body = lines[1].strip() if len(lines) > 1 else ""
            # Clean filename: just the base name without directory prefix
            filename = os.path.basename(name)
            documents.append({
                "filename": filename,
                "url": url,
                "content": body
            })

# Create a DataFrame
df = pd.DataFrame(documents)
print(f"Loaded {len(df)} documents")
print(f"\nColumns: {list(df.columns)}")
print(f"\n--- Content length stats (characters) ---")
df["content_length"] = df["content"].str.len()
print(df["content_length"].describe())

# Look at a few sample documents
for i in range(3):
    doc = df.iloc[i]
    print()
    print("=" * 80)
    print("FILE:", doc["filename"])
    print("URL: ", doc["url"])
    print("LENGTH:", doc["content_length"], "characters")
    print("-" * 80)
    preview = doc["content"][:500]
    print(preview)
    if len(doc["content"]) > 500:
        print("...")

df.head()

---

# 2. Chunk the Documents

Some of the pages could be too long to embed as a single unit. Down the line, the pages may be too long to stuff into the LLM's prompt during the generation step. As such, most of the RAG systems will break down big documents into into smaller **chunks**.

> 📚 **TERM: Chunking**  
> Splitting documents into smaller, self-contained pieces for embedding and retrieval. The goal is chunks that are small enough to be specific, but large enough to be meaningful.

Your task: **write a function that splits each document into chunks.**

Things to think about:
- What's a reasonable chunk size? (Think about what fits in a prompt vs. what's too vague)
- Should you split on sentences? Paragraphs? A fixed character/word count?
- Should chunks overlap? What happens if an answer spans two chunks?
- How do you keep track of which document each chunk came from? You may need that information down the line.

In [ ]:
def chunk_document(text, chunk_size=800, chunk_overlap=200):
    """
    Split a document into overlapping chunks.
    
    Strategy: split on paragraph boundaries (double newlines), then
    group paragraphs into chunks of roughly chunk_size characters.
    Adjacent chunks overlap by chunk_overlap characters to avoid
    losing context at boundaries.
    """
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    
    if not paragraphs:
        return [text] if text.strip() else []
    
    chunks = []
    current_chunk = []
    current_length = 0
    
    for para in paragraphs:
        para_len = len(para)
        
        if current_length + para_len > chunk_size and current_chunk:
            chunks.append("\n\n".join(current_chunk))
            
            # Build overlap: keep trailing paragraphs that fit within overlap size
            overlap_chunk = []
            overlap_length = 0
            for p in reversed(current_chunk):
                if overlap_length + len(p) <= chunk_overlap:
                    overlap_chunk.insert(0, p)
                    overlap_length += len(p)
                else:
                    break
            
            current_chunk = overlap_chunk
            current_length = overlap_length
        
        current_chunk.append(para)
        current_length += para_len
    
    if current_chunk:
        chunks.append("\n\n".join(current_chunk))
    
    return chunks


# Apply chunking to all documents
all_chunks = []

for _, row in df.iterrows():
    chunks = chunk_document(row["content"])
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "filename": row["filename"],
            "url": row["url"],
            "chunk_index": i,
            "chunk_text": chunk,
            "chunk_length": len(chunk)
        })

chunks_df = pd.DataFrame(all_chunks)
print(f"Total documents: {len(df)}")
print(f"Total chunks: {len(chunks_df)}")
print(f"Average chunks per document: {len(chunks_df) / len(df):.1f}")
print(f"\n--- Chunk length stats (characters) ---")
print(chunks_df["chunk_length"].describe())

# Preview chunks from first document
first_doc = chunks_df[chunks_df["filename"] == chunks_df.iloc[0]["filename"]]
for _, chunk_row in first_doc.iterrows():
    idx = chunk_row["chunk_index"]
    length = chunk_row["chunk_length"]
    print(f"\nChunk {idx} ({length} chars):")
    print(chunk_row["chunk_text"][:200])
    if length > 200:
        print("...")

---

# 3. Embed the Chunks

Now we need to turn each chunk into a vector so we can search over them. You've done this before in Lecture 4.

Your task: **embed all chunks using an embedding model.**

Tips:
- You could use a local model, or API model. What are the tradeoffs?
- This will take a while if you do it serially. You might want to use async/batch.
- Once you've created your embeddings, you may want to save them to disk so you don't have to redo this step every time
- You'll need to embed queries with the **same model** at search time

In [4]:
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
import pickle

load_dotenv(os.path.join("..", ".env"))
client = OpenAI()

EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDINGS_PATH = os.path.join("..", "data", "chunk_embeddings.pkl")
MAX_CHARS = 6000  # ~1500 tokens, safely under the 8192 token limit


def get_embeddings(texts, model=EMBEDDING_MODEL, batch_size=20):
    """
    Embed a list of texts using OpenAI's embedding API.
    Batches requests and truncates long texts to stay within token limits.
    """
    all_embeddings = []
    total_batches = (len(texts) - 1) // batch_size + 1
    for i in range(0, len(texts), batch_size):
        batch = [t[:MAX_CHARS] for t in texts[i:i + batch_size]]
        response = client.embeddings.create(model=model, input=batch)
        batch_embeddings = [item.embedding for item in response.data]
        all_embeddings.extend(batch_embeddings)
        print(f"  Embedded batch {i // batch_size + 1}/{total_batches}")
    return np.array(all_embeddings)


# Check if embeddings already exist on disk
if os.path.exists(EMBEDDINGS_PATH):
    print("Loading saved embeddings from disk...")
    with open(EMBEDDINGS_PATH, "rb") as f:
        saved = pickle.load(f)
    embeddings = saved["embeddings"]
    print(f"Loaded embeddings with shape: {embeddings.shape}")
else:
    print(f"Embedding {len(chunks_df)} chunks with {EMBEDDING_MODEL}...")
    chunk_texts = chunks_df["chunk_text"].tolist()
    embeddings = get_embeddings(chunk_texts)
    print(f"\nEmbeddings shape: {embeddings.shape}")
    
    # Save to disk so we don't have to redo this
    with open(EMBEDDINGS_PATH, "wb") as f:
        pickle.dump({"embeddings": embeddings}, f)
    print(f"Saved embeddings to {EMBEDDINGS_PATH}")


Embedding 59935 chunks with text-embedding-3-small...
  Embedded batch 1/2997
  Embedded batch 2/2997
  Embedded batch 3/2997
  Embedded batch 4/2997
  Embedded batch 5/2997
  Embedded batch 6/2997
  Embedded batch 7/2997
  Embedded batch 8/2997
  Embedded batch 9/2997
  Embedded batch 10/2997
  Embedded batch 11/2997
  Embedded batch 12/2997
  Embedded batch 13/2997
  Embedded batch 14/2997
  Embedded batch 15/2997
  Embedded batch 16/2997
  Embedded batch 17/2997
  Embedded batch 18/2997
  Embedded batch 19/2997
  Embedded batch 20/2997
  Embedded batch 21/2997
  Embedded batch 22/2997
  Embedded batch 23/2997
  Embedded batch 24/2997
  Embedded batch 25/2997
  Embedded batch 26/2997
  Embedded batch 27/2997
  Embedded batch 28/2997
  Embedded batch 29/2997
  Embedded batch 30/2997
  Embedded batch 31/2997
  Embedded batch 32/2997
  Embedded batch 33/2997
  Embedded batch 34/2997
  Embedded batch 35/2997
  Embedded batch 36/2997
  Embedded batch 37/2997
  Embedded batch 38/2997
  Emb

---

# 4. Retrieve

Now build the **R** in RAG. Given a user's question, find the most relevant chunks.

Your task: **write a retrieval function that takes a question and returns the most relevant chunks.**

Tips:
- You can use lexical or semantic search or both!
- How many chunks should you retrieve? Too few and you might miss the answer; too many and you'll overwhelm the LLM (and pay more tokens)
- Try a few test questions and eyeball whether the retrieved chunks are relevant
- Try a few questions and see what comes back. For example:
  - "What programs does the Gabelli School of Business offer?"
  - "How do I apply for financial aid?"
  - "Where is Fordham's campus?"

In [ ]:
def retrieve(question, top_k=5):
    """
    Given a user question, find the most relevant chunks using cosine similarity.
    
    Args:
        question: The user's question string
        top_k: Number of top chunks to return
    
    Returns:
        DataFrame of the top_k most relevant chunks with similarity scores
    """
    # Embed the question with the same model used for chunks
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=[question])
    query_embedding = np.array(response.data[0].embedding)
    
    # Compute cosine similarity between query and all chunk embeddings
    # cosine_sim = (A · B) / (||A|| * ||B||)
    norms = np.linalg.norm(embeddings, axis=1) * np.linalg.norm(query_embedding)
    similarities = embeddings @ query_embedding / norms
    
    # Get the top_k most similar chunk indices
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    # Build results DataFrame
    results = chunks_df.iloc[top_indices].copy()
    results["similarity"] = similarities[top_indices]
    return results


# Test with a few sample questions
test_questions = [
    "What programs does the Gabelli School of Business offer?",
    "How do I apply for financial aid?",
    "Where is Fordham's campus?",
]

for q in test_questions:
    print(f"\n{'=' * 80}")
    print(f"QUESTION: {q}")
    print(f"{'=' * 80}")
    results = retrieve(q)
    for _, row in results.iterrows():
        print(f"\n  [{row['similarity']:.3f}] {row['filename']}")
        print(f"  {row['chunk_text'][:150]}...")


---

# 5. Generate

Now build the **G** in RAG. Take the retrieved chunks and pass them to an LLM along with the user's question.

Your task: **write a function that takes a question and the retrieved chunks, builds a prompt, and calls an LLM to generate an answer.**

Tips:
- How should you structure the prompt? The LLM needs to know: (1) what is the context of the application, (2) what is the question, (3) what it should include in its answer
- What should the LLM do if the context doesn't contain the answer?
- Start with a cheap model; try a better one when you've figured out the pipeline

In [ ]:
def generate(question, context_chunks):
    """
    Given a question and retrieved chunks, build a prompt and call an LLM
    to generate an answer grounded in the provided context.
    
    Args:
        question: The user's question
        context_chunks: DataFrame of retrieved chunks (from retrieve())
    
    Returns:
        The LLM's answer as a string
    """
    # Build the context string from retrieved chunks
    context_parts = []
    for _, row in context_chunks.iterrows():
        source = row["url"]
        text = row["chunk_text"]
        context_parts.append(f"[Source: {source}]\n{text}")
    
    context = "\n\n---\n\n".join(context_parts)
    
    # Build the prompt
    system_prompt = (
        "You are a helpful assistant that answers questions about Fordham University. "
        "Use ONLY the provided context to answer the question. "
        "If the context does not contain enough information to answer, say so honestly. "
        "Cite the source URLs when possible."
    )
    
    user_prompt = f"Context:\n{context}\n\n---\n\nQuestion: {question}"
    
    # Call the LLM
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.2,
    )
    
    return response.choices[0].message.content


# Test it out
question = "How do I apply for financial aid at Fordham?"
results = retrieve(question)
answer = generate(question, results)

print(f"QUESTION: {question}\n")
print(f"ANSWER:\n{answer}")


---

# 6. Wire everything together

Combine the previous steps into a simple function that takes in a question and returns an answer.

Your task: **write a `rag(question)` function that retrieves relevant chunks and generates an answer.**

In [ ]:
def rag(question, top_k=5):
    """
    Full RAG pipeline: retrieve relevant chunks and generate an answer.
    
    Args:
        question: The user's question about Fordham University
        top_k: Number of chunks to retrieve
    
    Returns:
        The generated answer string
    """
    # Retrieve
    relevant_chunks = retrieve(question, top_k=top_k)
    
    # Generate
    answer = generate(question, relevant_chunks)
    
    return answer


# Try it out with a few questions
questions = [
    "What programs does the Gabelli School of Business offer?",
    "Where is Fordham located?",
    "What is the tuition for graduate students?",
]

for q in questions:
    print()
    print("=" * 80)
    print("Q:", q)
    print("=" * 80)
    print(rag(q))


---

# 7. Evaluate, experiment and improve

Your RAG system works — but there's always room to make it better. 

Your task: **evaluate, experiment, and improve your system**

Tips:
- How do you know that your system is working or that your changes are improving it?
- Try different questions — where does it do well? Where does it struggle?
- Adjust the number of retrieved chunks — what happens with more or fewer?
- Try different chunking strategies — bigger chunks? Smaller? Overlap?
- Try a different embedding model — does it change retrieval quality?
- Improve the prompt — can you get better, more concise answers?
- Add source attribution — can the system tell the user which pages the answer came from?

In [ ]:
# Placeholder for your implementation

---

# 8. (Optional) Make it an app

So far your RAG system lives inside a notebook. That's great for development — but nobody is going to use your Jupyter notebook to ask questions about Fordham. Let's turn it into a real web app.

> 📚 **TERM: Streamlit**  
> A Python library that turns plain Python scripts into interactive web apps. You write Python — no HTML, CSS, or JavaScript — and Streamlit renders it as a web page with inputs, buttons, and formatted output. It's the fastest way to go from "I have a function" to "I have a web app."

Your task: **create a Streamlit app that lets a user type a question about Fordham and get an answer from your RAG system.**

To get started:
- Install it: `uv pip install streamlit` 
- A Streamlit app is just a `.py` file (not a notebook). Create something like `fordham_rag_app.py`
- Run it: `streamlit run scripts/fordham_rag_app.py` — this opens a browser tab with your app

Tips:
- Check out the [Streamlit docs](https://docs.streamlit.io/) — the "Get started" tutorial is very short
- Your best bet is to vibecode your way to this. You'll be surprised how fast you can get it up and running

---

# Summary

## What You Built

| Step | What You Did | What It Does |
|------|-------------|-------------|
| **Load** | Read 9,500+ Fordham web pages | Get raw content |
| **Chunk** | Split pages into smaller pieces | Make content searchable and promptable |
| **Embed** | Turn chunks into vectors | Enable semantic search |
| **Retrieve** | Find relevant chunks for a question | The **R** in RAG |
| **Generate** | Ask an LLM to answer using the chunks | The **G** in RAG |
| **RAG** | Wire it all together | Question in, answer out |

## The Big Picture

RAG is one of the most common patterns in AI engineering today. What you built here is the same core architecture behind tools like ChatGPT with search, Perplexity, enterprise Q&A bots, and more. The details get more sophisticated (vector databases, reranking, query rewriting, evaluation) but the pattern is the same:

**Find relevant stuff → give it to an LLM → get an answer.**

You can just build things.